<a href="https://colab.research.google.com/github/HillaryDrugs/li7/blob/main/OpenAi_Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install openai-whisper datasets jiwer numpy torch soundfile torchaudio huggingface_hub

import os
import re
import torch
import numpy as np
import soundfile as sf
import torchaudio
import whisper

from datasets import load_dataset, Audio
from jiwer import process_words
from huggingface_hub import hf_hub_download

# -----------------------------
# 0) Setup
# -----------------------------
DATASET_ID = "NightPrince/MasriSpeech-Full"
NUM_SAMPLES = 500
TARGET_SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# 1) Arabic normalization
# -----------------------------
def normalize_ar(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[\u064B-\u0652]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'[ؤئ]', 'ء', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'[^\u0621-\u064A\s\d]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# -----------------------------
# 2) Load dataset (NO decoding)
# -----------------------------
print("Loading dataset...")
ds = load_dataset(DATASET_ID, split="validation")
ds = ds.cast_column("audio", Audio(decode=False))  # IMPORTANT: avoid torchcodec
ds = ds.select(range(min(NUM_SAMPLES, len(ds))))

# Normalize transcriptions (no multiprocessing)
refs_norm = [normalize_ar(ex["transcription"]) for ex in ds]

print("Samples used:", len(ds))

# -----------------------------
# 3) Audio loader: local cache path OR bytes fallback
# -----------------------------
_resamplers = {}

def load_audio_np_from_audioobj(audio_obj):
    """
    Works without torchcodec:
    - If audio_obj contains local 'path' and it exists -> read it
    - Else if it contains 'bytes' -> read from bytes
    - Else -> raise
    """
    if not isinstance(audio_obj, dict):
        raise ValueError(f"Unexpected audio object type: {type(audio_obj)}")

    # (A) Prefer bytes if present (most robust when available)
    if audio_obj.get("bytes", None) is not None:
        import io
        wav, sr = sf.read(io.BytesIO(audio_obj["bytes"]))
    else:
        p = audio_obj.get("path", "")
        if p and os.path.exists(p):
            wav, sr = sf.read(p)
        else:
            # If neither bytes nor a valid local path exists, decoding isn't possible without torchcodec
            raise FileNotFoundError(
                "Audio is not available as local path or bytes. "
                "This dataset requires torchcodec for decoding in this environment."
            )

    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)

    if sr != TARGET_SR:
        if sr not in _resamplers:
            _resamplers[sr] = torchaudio.transforms.Resample(sr, TARGET_SR)
        wav = _resamplers[sr](torch.tensor(wav).unsqueeze(0)).squeeze(0).numpy().astype(np.float32)

    return wav

# -----------------------------
# 4) Load Whisper
# -----------------------------
WHISPER_MODEL_NAME = "medium"
print("Loading Whisper:", WHISPER_MODEL_NAME)
model = whisper.load_model(WHISPER_MODEL_NAME).to(device)
print("Model loaded.")

# -----------------------------
# 5) WER breakdown (your format)
# -----------------------------
def wer_breakdown(refs, hyps):
    total_words = 0
    total_del = 0
    total_sub = 0
    perfect = 0

    for r, h in zip(refs, hyps):
        m = process_words(r, h)
        n = m.hits + m.substitutions + m.deletions
        total_words += n
        total_del += m.deletions
        total_sub += m.substitutions
        if (m.substitutions + m.deletions + m.insertions) == 0:
            perfect += 1

    perfect_wer  = 1 - (perfect / len(refs)) if refs else 0.0
    deletion_wer = (total_del / total_words) if total_words else 0.0
    typo_wer     = (total_sub / total_words) if total_words else 0.0

    print(f"Perfect WER: {perfect_wer:.1f}")
    print(f"Deletion WER: {deletion_wer:.1f}")
    print(f"Typo WER: {typo_wer:.1f}")

# -----------------------------
# 6) Evaluate 500 samples
# -----------------------------
hyps = []
refs = []

print("\nRunning evaluation...")
for i in range(len(ds)):
    ex = ds[i]
    ref = refs_norm[i]
    if not ref:
        continue

    wav = load_audio_np_from_audioobj(ex["audio"])

    with torch.no_grad():
        out = model.transcribe(wav, language="ar", fp16=(device == "cuda"), verbose=False)

    hyp = normalize_ar(out.get("text", ""))

    refs.append(ref)
    hyps.append(hyp)

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{len(ds)}")

print("\n==============================")
print("FINAL RESULTS")
print("==============================")
wer_breakdown(refs, hyps)
